In [ ]:
import pandas as pd
import numpy as np
import string
import random
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

# Classical Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Modern Deep Learning
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, LSTM
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# STAGE 1: DATA PREPARATION & AUGMENTATION (Char-CNN Proposal Setup)

print("Preparing dataset and applying augmentation...")
df = pd.read_excel('combined_dataset.xlsx', sheet_name='Sheet1')
df_clean = df.dropna(subset=['konten', 'spam']).copy()
df_clean['spam'] = df_clean['spam'].str.strip().str.upper()

X_raw = df_clean['konten'].astype(str).tolist()
y = df_clean['spam'].map({'SPAM': 1, 'HAM': 0}).values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

X_train_spam = [text for text, label in zip(X_train_raw, y_train) if label == 1]
X_train_ham = [text for text, label in zip(X_train_raw, y_train) if label == 0]

X_train_augmented, y_train_augmented = [], []

def apply_obfuscation(text, technique):
    text = text.lower()
    if technique == 'substitution': return text.translate(str.maketrans('aeio', '4310'))
    elif technique == 'insertion': return ' '.join(['.'.join(list(w)) if len(w) > 2 else w for w in text.split()])
    elif technique == 'vowel_removal': return text.translate(str.maketrans('', '', 'aeiou'))
    return text

# SPAM Injection
for text in X_train_spam:
    X_train_augmented.extend([text, text])
    y_train_augmented.extend([1, 1])
    X_train_augmented.append(apply_obfuscation(text, 'substitution')); y_train_augmented.append(1)
    X_train_augmented.append(apply_obfuscation(text, 'insertion')); y_train_augmented.append(1)
    X_train_augmented.append(apply_obfuscation(text, 'vowel_removal')); y_train_augmented.append(1)

# HAM Injection (To balance dataset and prevent numerical bias)
target_ham_count = len(X_train_spam) * 5
for _ in range(target_ham_count):
    ham_text = random.choice(X_train_ham)
    if random.random() > 0.4:
        rand_num = random.randint(1, 99)
        ham_text += random.choice([f" jam {rand_num}", f" tgl {rand_num}", f" {rand_num} org", ""]) + random.choice([" ya", " nih", " bro", " pak", ""])
    X_train_augmented.append(ham_text)
    y_train_augmented.append(0)

indices = np.arange(len(X_train_augmented))
np.random.shuffle(indices)
X_train_augmented = [X_train_augmented[i] for i in indices]
y_train_augmented = np.array([y_train_augmented[i] for i in indices])

valid_alphabet = string.ascii_lowercase + string.digits + " .,?!"
def clean_text(text): return ''.join([char for char in text.lower() if char in valid_alphabet])

X_train_clean = [clean_text(sms) for sms in X_train_augmented]
X_test_clean = [clean_text(sms) for sms in X_test_raw]


# OUTPUT 1: SUMMARY STATISTIC
df_stats = pd.DataFrame({'content': X_train_augmented + X_test_raw, 'label': list(y_train_augmented) + list(y_test)})
print("\n" + "="*50)
print("OUTPUT 1: FINAL DATASET SUMMARY STATISTICS")
print("="*50)
print(f"Total Messages        : {len(df_stats)} rows")
print(f"Unique Messages       : {df_stats['content'].nunique()} rows")
print(f"Duplicate Messages    : {len(df_stats) - df_stats['content'].nunique()} rows")
print(f"  * Total Trained SPAM: {sum(df_stats['label'] == 1)} rows")
print(f"  * Total Trained HAM : {sum(df_stats['label'] == 0)} rows")
print("="*50 + "\n")


# STAGE 2: FEATURE EXTRACTION & BASELINE TRAINING (FAIR COMPARISON)
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_test_tfidf = tfidf.transform(X_test_clean)

MAX_LEN_WORD = 100
word_tokenizer = Tokenizer(num_words=5000)
word_tokenizer.fit_on_texts(X_train_clean)
X_train_word = pad_sequences(word_tokenizer.texts_to_sequences(X_train_clean), maxlen=MAX_LEN_WORD, padding='pre')
X_test_word = pad_sequences(word_tokenizer.texts_to_sequences(X_test_clean), maxlen=MAX_LEN_WORD, padding='pre')

print("Training Baseline Models (TF-IDF & Word-Level)...")
nb = MultinomialNB(); nb.fit(X_train_tfidf, y_train_augmented)
dt = DecisionTreeClassifier(random_state=42); dt.fit(X_train_tfidf, y_train_augmented)
svm = SVC(probability=True, kernel='linear', random_state=42); svm.fit(X_train_tfidf, y_train_augmented)
rf = RandomForestClassifier(n_estimators=100, random_state=42); rf.fit(X_train_tfidf, y_train_augmented)
lr = LogisticRegression(random_state=42); lr.fit(X_train_tfidf, y_train_augmented)

es = EarlyStopping(monitor='loss', patience=2, restore_best_weights=True)
inp_w_cnn = Input(shape=(MAX_LEN_WORD,))
x = Embedding(5000, 50)(inp_w_cnn); x = Conv1D(128, 3, activation='relu')(x); x = GlobalMaxPooling1D()(x)
out_w_cnn = Dense(1, activation='sigmoid')(Dropout(0.5)(Dense(64, activation='relu')(x)))
word_cnn = Model(inp_w_cnn, out_w_cnn); word_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
word_cnn.fit(X_train_word, y_train_augmented, epochs=5, batch_size=32, callbacks=[es], verbose=0)

inp_w_lstm = Input(shape=(MAX_LEN_WORD,))
x = Embedding(5000, 50)(inp_w_lstm); x = LSTM(32)(x)
out_w_lstm = Dense(1, activation='sigmoid')(Dropout(0.2)(Dense(32, activation='relu')(x)))
word_lstm = Model(inp_w_lstm, out_w_lstm); word_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
word_lstm.fit(X_train_word, y_train_augmented, epochs=5, batch_size=32, callbacks=[es], verbose=0)

# Load the previously trained Char-CNN Tokenizer and Model
with open('tokenizer_char.pkl', 'rb') as handle: char_tokenizer = pickle.load(handle)
char_cnn = load_model('char_cnn_spam_model.keras')
X_test_char = pad_sequences(char_tokenizer.texts_to_sequences(X_test_clean), maxlen=500, padding='post', truncating='post')

# OUTPUT 2: EVALUATION & BAR CHART
eval_results = {'Model': [], 'Accuracy': [], 'F1-Score': []}
model_list = [
    ("Naive Bayes", nb, 'classical'), ("Decision Tree", dt, 'classical'), ("SVM (Linear)", svm, 'classical'),
    ("Random Forest", rf, 'classical'), ("Logistic Reg", lr, 'classical'), ("Word-CNN", word_cnn, 'modern'),
    ("Word-LSTM", word_lstm, 'modern'), ("Char-CNN (Proposed)", char_cnn, 'proposed')
]

for name, model, m_type in model_list:
    if m_type == 'classical': y_pred = model.predict(X_test_tfidf)
    elif m_type == 'modern': y_pred = (model.predict(X_test_word, verbose=0) > 0.5).astype(int)
    else: y_pred = (model.predict(X_test_char, verbose=0) > 0.5).astype(int)

    eval_results['Model'].append(name)
    eval_results['Accuracy'].append(accuracy_score(y_test, y_pred))
    eval_results['F1-Score'].append(f1_score(y_test, y_pred))

df_eval = pd.DataFrame(eval_results)
print("="*65)
print("OUTPUT 2: PERFORMANCE COMPARISON OF 8 MODELS ON TEST DATA")
print("="*65)
print(df_eval.to_string(index=False))
print("="*65)

# Plotting the Bar Chart
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
df_melted = pd.melt(df_eval, id_vars=['Model'], value_vars=['Accuracy', 'F1-Score'], var_name='Metric', value_name='Score')
ax = sns.barplot(x='Model', y='Score', hue='Metric', data=df_melted, palette='viridis')
plt.title('Accuracy and F1-Score Comparison of 8 Classification Models', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Classification Models', fontsize=12, fontweight='bold')
plt.ylabel('Performance Score', fontsize=12, fontweight='bold')
plt.ylim(0.85, 1.05); plt.xticks(rotation=45, ha='right', fontsize=11)
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.3f'), (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', xytext=(0, 9), textcoords='offset points', fontsize=9)
plt.tight_layout(); plt.show()


# STAGE 3: ADVERSARIAL ROBUSTNESS PROOF (CHECKMATE SCENARIOS)
proof_messages = [
    ("Message 1 (Extreme OOV Spam & Numerical Bias)", "s.3.l.4.m.a.t n.0.m.0.r a.n.d.a m.3.m.3.n.4.n.g.k.4.n 10 j.u.t.a", 1),
    ("Message 2 (Normal Text / False Positive Trap)", "Tolong belikan martabak manis rasa keju coklat di simpang depan ya, jangan lupa bawa payung karena hujan turun deras.", 0)
]

proof_results = {'Model': [], 'Prediction 1 (Target: SPAM)': [], 'Prediction 2 (Target: HAM)': []}

for name, model, m_type in model_list:
    pred_row = []
    for _, text, target in proof_messages:
        clean_msg = clean_text(text)
        if m_type == 'classical': prob = model.predict_proba(tfidf.transform([clean_msg]))[:, 1][0]
        elif m_type == 'modern': prob = model.predict(pad_sequences(word_tokenizer.texts_to_sequences([clean_msg]), maxlen=MAX_LEN_WORD, padding='pre'), verbose=0)[0][0]
        else: prob = model.predict(pad_sequences(char_tokenizer.texts_to_sequences([clean_msg]), maxlen=500, padding='post', truncating='post'), verbose=0)[0][0]

        verdict = "SPAM" if prob > 0.5 else "HAM"
        status = "SUCCESS" if (prob > 0.5 and target == 1) or (prob <= 0.5 and target == 0) else "FAILED"
        pred_row.append(f"{verdict} ({status})")

    proof_results['Model'].append(name)
    proof_results['Prediction 1 (Target: SPAM)'].append(pred_row[0])
    proof_results['Prediction 2 (Target: HAM)'].append(pred_row[1])

# OUTPUT 3: BASELINE VULNERABILITY PROOF TABLE
df_proof = pd.DataFrame(proof_results)
print("\n" + "="*95)
print("OUTPUT 3: ADVERSARIAL ROBUSTNESS PROOF USING 2 SPECIFIC MESSAGES")
print("="*95)
print(f"Message 1 (Target SPAM) : '{proof_messages[0][1]}'")
print(f"Message 2 (Target HAM)  : '{proof_messages[1][1]}'\n")
print(df_proof.to_string(index=False))
print("="*95)